In [7]:
%pip install openpyxl

   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   --------- ------------------------------ 61.4/250.9 kB 3.4 MB/s eta 0:00:01
   --------------------------- ------------ 174.1/250.9 kB 1.7 MB/s eta 0:00:01
   --------------------------- ------------ 174.1/250.9 kB 1.7 MB/s eta 0:00:01
   --------------------------- ------------ 174.1/250.9 kB 1.7 MB/s eta 0:00:01
   --------------------------- ------------ 174.1/250.9 kB 1.7 MB/s eta 0:00:01
   ---------------------------------- --- 225.3/250.9 kB 811.5 kB/s eta 0:00:01
   -------------------------------------- 250.9/250.9 kB 810.3 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import os

CHAPTER 1: CONCATENATE DATASETS

In [ ]:
[number for number in range(1, 10)]

In [188]:
# COLLECT DATASET
dataset_list = [file for file in os.listdir("../dataset/nasa/raw") if file.startswith("DL_FIRE_") ]
name_list = ["nasa1_archive", "nasa1_nrt"]

print("DATASET LIST: ")
for i, f in enumerate(dataset_list) : print(f"{i}. {f}")

os.listdir(f"../dataset/nasa/raw/{dataset_list[1-1]}")

DATASET LIST: 
0. DL_FIRE_J1V-C2_838646
1. DL_FIRE_J1V-C2_838649
2. DL_FIRE_J1V-C2_838655
3. DL_FIRE_J2V-C2_838647
4. DL_FIRE_J2V-C2_838650
5. DL_FIRE_J2V-C2_838656
6. DL_FIRE_SV-C2_838648
7. DL_FIRE_SV-C2_838651
8. DL_FIRE_SV-C2_838657


['fire_archive_J1V-C2_838646.csv', 'fire_nrt_J1V-C2_838646.csv']

In [ ]:
# COLLECT DATASET
dataset_list = [file for file in os.listdir("../dataset/nasa/raw") if file.startswith("DL_FIRE_") ]
name_list = ["nasa1_archive", "nasa1_nrt"]

# GLOBAL DATASET STORAGE
# key --> download id 
# value --> the entire df
dataset_storage = {}

# LOAD DATASETS:
for i, f in enumerate(dataset_list):
    print(f"Processing dataset in folder... {i}. {f}")

    for j, k in enumerate(os.listdir(f"../dataset/nasa/raw/{dataset_list[i]}")):
        key = os.listdir(f"../dataset/nasa/raw/{dataset_list[i]}")[j].split(".")[0]
        value = pd.read_csv(f"../dataset/nasa/raw/{dataset_list[i]}/{os.listdir(f"../dataset/nasa/raw/{dataset_list[i]}")[j]}")

        # ADDING COUNTRY NAME COLUMN (According to Readme detail)
        if '838650' in key or '838649' in key or '838651' in key:
            value['domain'] = pd.Series(['Malaysia' for x in range(len(value.index))])
        elif '838657' in key or '838656' in key or '838655' in key:
            value['domain'] = pd.Series(['Singapore' for x in range(len(value.index))])
        elif '838647' in key or '838648' in key or '838646' in key:
            value['domain'] = pd.Series(['Indonesia' for x in range(len(value.index))])
        else: 
            print("Unknown ID Found.")
            print(key)

        
        # store to dict
        dataset_storage[key] = value
    


Processing dataset in folder... 0. DL_FIRE_J1V-C2_838646
Processing dataset in folder... 1. DL_FIRE_J1V-C2_838649
Processing dataset in folder... 2. DL_FIRE_J1V-C2_838655
Processing dataset in folder... 3. DL_FIRE_J2V-C2_838647
Processing dataset in folder... 4. DL_FIRE_J2V-C2_838650
Processing dataset in folder... 5. DL_FIRE_J2V-C2_838656
Processing dataset in folder... 6. DL_FIRE_SV-C2_838648
Processing dataset in folder... 7. DL_FIRE_SV-C2_838651
Processing dataset in folder... 8. DL_FIRE_SV-C2_838657


In [193]:
# CHECK KEYS
a = list(dataset_storage.keys())

print("\n STORAGE KEY LIST:")
for i, f in enumerate(a) : print(f"{i}. {f}")


 STORAGE KEY LIST:
0. fire_archive_J1V-C2_838646
1. fire_nrt_J1V-C2_838646
2. fire_archive_J1V-C2_838649
3. fire_nrt_J1V-C2_838649
4. fire_nrt_J1V-C2_838655
5. fire_archive_J2V-C2_838647
6. fire_nrt_J2V-C2_838647
7. fire_archive_J2V-C2_838650
8. fire_nrt_J2V-C2_838650
9. fire_archive_J2V-C2_838656
10. fire_nrt_J2V-C2_838656
11. fire_archive_SV-C2_838648
12. fire_nrt_SV-C2_838648
13. fire_archive_SV-C2_838651
14. fire_nrt_SV-C2_838651
15. fire_archive_SV-C2_838657
16. fire_nrt_SV-C2_838657


In [ ]:
#combining archive with nrt
j = 0
val_dataset_storage = list(dataset_storage.values())

#subdatasets are all datasets in all domains
subdatasets = val_dataset_storage


# Combine both dataframes
combined = pd.concat(subdatasets, ignore_index=True)

# 1. Check domain counts right after your loop / concatenation (before any sorting or dropping)
print("--- BEFORE DEDUPLICATION ---")
print(combined['domain'].value_counts(dropna=False))

# 2. Run your priority sort and drop_duplicates
combined['_priority'] = (combined['domain'] == 'Indonesia') & (combined['version'] == '2.0NRT')
combined = combined.sort_values(by='_priority', ascending=False)

cleaned_data = combined.drop_duplicates(
    subset=['latitude', 'longitude', 'acq_date', 'acq_time'], 
    keep='first'
)

# 3. Check domain counts immediately after drop_duplicates
print("\n--- AFTER DROP_DUPLICATES ---")
print(cleaned_data['domain'].value_counts(dropna=False))


# 4. Drop the temporary helper column
cleaned_data = cleaned_data.drop(columns=['_priority'])

#save individual concatenated datasets
os.makedirs("../dataset/nasa/concatenated", exist_ok=True)
cleaned_data.to_csv(f"../dataset/nasa/concatenated/nasa_concatenated.csv", index=False)
print(f"done nasa_concatenated")



--- BEFORE DEDUPLICATION ---
domain
Indonesia    1819668
Malaysia       11243
Singapore          7
Name: count, dtype: int64

--- AFTER DROP_DUPLICATES ---
domain
Indonesia    1819667
Malaysia       11243
Singapore          7
Name: count, dtype: int64
done nasa_concatenated


CHAPTER 2: CLEANING DATASETS

In [233]:
# Load the .csv file from concatenated folder
nasa_df = pd.read_csv("../dataset/nasa/concatenated/nasa_concatenated.csv")

nasa_df.head()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_39168\2002804258.py:2: DtypeWarning: Columns (10: version) have mixed types. Specify dtype option on import or set low_memory=False.
  nasa_df = pd.read_csv("../dataset/nasa/concatenated/nasa_concatenated.csv")


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,domain
0,-2.83402,107.77584,309.89,0.39,0.36,2026-09-17,1811,N21,VIIRS,n,2.0NRT,292.29,1.15,N,NaN,Indonesia
1,-2.30088,132.56708,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.48,16.15,D,NaN,Indonesia
2,-2.29571,132.50427,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.43,12.30,D,NaN,Indonesia
3,-2.28784,132.55690,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.32,24.59,D,NaN,Indonesia
4,-2.28724,132.56094,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.39,9.35,D,NaN,Indonesia


In [228]:
# Check missing values count and percentage per column
print("checking...")
missing_count = nasa_df.isna().sum()
missing_pct = nasa_df.isna().mean() * 100

summary_df = pd.DataFrame(
    {'Missing Count': missing_count, 'Missing Pct (%)': missing_pct}
)
print(summary_df.sort_values(by='Missing Pct (%)', ascending=False), "\n")

checking...
            Missing Count  Missing Pct (%)
type              1728742        94.419463
latitude                0         0.000000
brightness              0         0.000000
longitude               0         0.000000
track                   0         0.000000
acq_date                0         0.000000
acq_time                0         0.000000
scan                    0         0.000000
satellite               0         0.000000
instrument              0         0.000000
version                 0         0.000000
confidence              0         0.000000
bright_t31              0         0.000000
frp                     0         0.000000
daynight                0         0.000000
domain                  0         0.000000 



In [229]:
nasa_df.iloc[:4]

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,domain
0,-2.83402,107.77584,309.89,0.39,0.36,2026-09-17,1811,N21,VIIRS,n,2.0NRT,292.29,1.15,N,NaN,Indonesia
1,-2.30088,132.56708,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.48,16.15,D,NaN,Indonesia
2,-2.29571,132.50427,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.43,12.30,D,NaN,Indonesia
3,-2.28784,132.55690,367.00,0.45,0.39,2026-10-17,423,N21,VIIRS,h,2.0NRT,294.32,24.59,D,NaN,Indonesia


In [234]:
nasa_cleaned = []

print("CLEANING...")

#check length
print("Row Length prior:", len(nasa_df))

#drop column "type"
print("Removing Column 'type'...")
#column before
print("column before: ", nasa_df.columns.tolist())
# Drop a single column
df = nasa_df.drop(columns='type')
#column after
print("column after: ", df.columns.tolist())

# remove confidence low (l)
print("Filtering Confidence...")
# removing row which column 'confidence' has cell value 'l' 
df = df[df["confidence"] != "l"]
print("Row Length after confidence filtering: ", len(df))

# remove Nan
print("Removing Nan...")
df = df.dropna()
#check aftermath length
print("Row Length after Nan removal:", len(df))

# 2. Convert Date column to datetime type
df['acq_date'] = pd.to_datetime(df['acq_date'])

# 3. Sort by Date first, then by Time
sorted = df.sort_values(by=['acq_date', 'acq_time'], ascending=[True, True])


#saving to .csv
os.makedirs("../dataset/nasa/cleaned", exist_ok=True)
sorted.to_csv(f"../dataset/nasa/cleaned/nasa_cleaned.csv", index=False)
print("Dataset saved to ../dataset/nasa/cleaned/nasa_cleaned.csv")


CLEANING...
Row Length prior: 1830917
Removing Column 'type'...
column before:  ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'domain']
column after:  ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'domain']
Filtering Confidence...
Row Length after confidence filtering:  1734047
Removing Nan...
Row Length after Nan removal: 1734047
Dataset saved to ../dataset/nasa/cleaned/nasa_cleaned.csv
